In [1]:
!git clone https://github.com/nickkats1/Prompting

Cloning into 'Prompting'...
remote: Enumerating objects: 181, done.
remote: Counting objects: 100% (181/181), done.
remote: Compressing objects: 100% (136/136), done.
remote: Total 181 (delta 65), reused 150 (delta 37), pack-reused 0 (from 0)
Receiving objects: 100% (181/181), 1.79 MiB | 29.53 MiB/s, done.
Resolving deltas: 100% (65/65), done.


# Small LLM From Scratch (Base Model)

I will be using the GPT2 varient of the transformer. Most modern LLM's do not use this. I am not going to show full seq-to-seq due to the inconvenience

In [1]:
import torch
import torch.nn as nn
import tiktoken
from pathlib import Path
import sys
sys.path.append("/content/Prompting")

data_path = Path("/content/training_data.txt")


text_data = data_path.read_text(encoding="utf-8", errors="ignore")
tokenizer = tiktoken.get_encoding("gpt2")

print(f"Training data loaded from {data_path}")

Training data loaded from /content/training_data.txt


In [2]:
total_chars = len(text_data)
print("total characters: ")
print(total_chars)
print("total tokens")
print(len(tokenizer.encode(text_data)))

total characters: 
55722490
total tokens
16581742


## Splitting the data

This has 16581742 tokens. Splitting the data is crucial for evaluating the model. This does not have to be said

In [3]:
train_ratio = 0.8
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]

In [4]:
print("length of training data")
print(len(train_data))
print()
print("length of val data")
print()
print(len(val_data))

length of training data
44577992

length of val data

11144498


### Put data into GPTDataset and then put that data into dataloader

In [5]:
from transformer.dataset import create_dataloader
from transformer.config import CONFIG

train_loader = create_dataloader(
    train_data,
    batch_size=8,
    max_length=CONFIG["context_length"],
    stride=CONFIG["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader(
    val_data,
    batch_size=8,
    max_length=CONFIG["context_length"],
    stride=CONFIG["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

In [6]:
next(iter(train_loader))

[tensor([[   19,  8141,     0,  ...,   994,  1058,   737],
         [ 3383,   471,    13,  ...,  1300,  1303, 24095],
         [ 2582,   355,  1248,  ...,   198,   198, 49945],
         ...,
         [ 5643,   198,   198,  ...,   536,   667,    88],
         [  198,   259, 11291,  ...,   452,  2929,    13],
         [29617,   519, 20996,  ..., 34296,  5277,   198]]),
 tensor([[ 8141,     0,  1303,  ...,  1058,   737,   198],
         [  471,    13,    50,  ...,  1303, 24095,   500],
         [  355,  1248,  5556,  ...,   198, 49945,    25],
         ...,
         [  198,   198, 42787,  ...,   667,    88,   329],
         [  259, 11291,  1105,  ...,  2929,    13,   198],
         [  519, 20996,   745,  ...,  5277,   198,  1324]])]

In [7]:
next(iter(val_loader))

[tensor([[34409,  1366,    11,  ...,   262,  6617,  8991],
         [   13,   198,  1858,  ...,    17,  4880,    75],
         [  198,   198,    59,  ..., 11291,  1105,    13],
         ...,
         [   83,   256,   532,  ..., 21862,   896,    11],
         [  412,    13,   357,  ...,    11,   311,  1539],
         [  347,  5540,    11,  ...,   584,  1181,    12]]),
 tensor([[ 1366,    11,   198,  ...,  6617,  8991,    13],
         [  198,  1858,   389,  ...,  4880,    75,   198],
         [  198,    59,   198,  ...,  1105,    13,    23],
         ...,
         [  256,   532, 19242,  ...,   896,    11,   412],
         [   13,   357, 23847,  ...,   311,  1539,   347],
         [ 5540,    11,   449,  ...,  1181,    12,  2673]])]

In [8]:
from torch.cuda import is_available
from transformer.utils import calc_loss_batch, calc_loss_loader
from transformer.model import GPTModel
device = ("cuda:0" if torch.cuda.is_available() else "cpu")
model = GPTModel(cfg=CONFIG)
model.to(device)

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (attn): SelfAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=False)
        (W_key): Linear(in_features=768, out_features=768, bias=False)
        (W_value): Linear(in_features=768, out_features=768, bias=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (attn): SelfAttention(
        (W_query): Linear(in_features=768, out_featur

In [9]:
from transformer.trainer import train
torch.manual_seed(123)

model = GPTModel(CONFIG)
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=0.1)

num_epochs = 10
train_losses, val_losses, tokens_seen = train(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context="Winston Reed", tokenizer=tokenizer
)

Ep 1 (Step 000000): Train loss 10.095, Val loss 10.198
Winston Reed                                                  
Ep 1 (Step 000001): Train loss 10.095, Val loss 10.198
Winston Reed                                                  
Ep 1 (Step 000002): Train loss 10.095, Val loss 10.198
Winston Reed.                                                 
Ep 1 (Step 000003): Train loss 10.095, Val loss 10.198
Winston Reed..                                                
Ep 1 (Step 000004): Train loss 10.095, Val loss 10.198
Winston Reed.                                                 
Ep 1 (Step 000005): Train loss 8.959, Val loss 9.110
Winston Reed                                                  
Ep 1 (Step 000006): Train loss 8.959, Val loss 9.110
Winston Reed                                                  
Ep 1 (Step 000007): Train loss 8.959, Val loss 9.110
Winston Reed                                                  
Ep 1 (Step 000008): Train loss 8.959, Val loss 9.110
Winston R

### About the output

This over-fit a good amount. This is not a easy task at all and the quality of the data and preparing it is as important as anything else.

### Generating samples

In [11]:
from transformer.utils import generate, token_ids_to_text, text_to_token_ids
from transformer.config import CONFIG

inference_device = torch.device("cuda")

torch.manual_seed(123)

token_ids = generate(
    model=model,
    idx=text_to_token_ids("Winston Reed", tokenizer).to(inference_device),
    max_new_tokens=35,
    context_size=CONFIG["context_length"],
    top_k=80,
    temperature=1.4
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 Winston Reed & U.S. Supreme Court -Lyi Group.UL . .�
Nul Aries,  $💿�
The truoclast


## Total Parameters


In [12]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total Parameters: {total_params:,}")

Total Parameters: 163,009,536


In [13]:
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    },
    "model_and_optimizer.pth"
)